# Lidl España — extractor corregido (v3)

**Objetivo:** recuperar productos útiles para comparar cestas, sin incorporar artículos que no coincidan con la búsqueda ni inventar precios unitarios. Basado en el JSON observado en `lidl_api_corregido(1).ipynb`.

**Importante:** HTTP 200 no demuestra que los precios correspondan a la tienda física del usuario. `numFound` incluye resultados del buscador que podrían no ser fichas de producto. Se guardan registros rechazados y diagnósticos para auditoría.

Ejecuta las celdas en orden. Primero `TEST_MODE=True`; cuando termine correctamente, pasa a `False`. No cambies `USE_STRICT_FILTER` salvo que revises manualmente los resultados.

In [1]:
# %pip install requests pandas
import requests, pandas as pd, json, re, time, unicodedata, html
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import urljoin
from IPython.display import display

BASE_URL = "https://www.lidl.es"
API_URL = BASE_URL + "/q/api/search"
SEARCH_TERMS = ["leche", "huevos", "manzana", "plátano", "tomate", "arroz", "pasta", "pan", "pollo", "carne picada", "salmón", "yogur", "queso", "agua", "zumo", "atún", "garbanzos", "aceite", "detergente", "papel higiénico"]
TEST_MODE = True
TERMS = ["leche"] if TEST_MODE else SEARCH_TERMS
PAGE_SIZE = 10              # El tamaño que obtuvo 200 en tu prueba
MAX_PAGES = 15              # Límite de seguridad; ajustable según la API
DELAY = 0.7
MAX_RETRIES = 3
TIMEOUT = 20
USE_STRICT_FILTER = True

cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name == "api_clients" else cwd
RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
for directory in (RAW_DIR, PROCESSED_DIR): directory.mkdir(parents=True, exist_ok=True)
print("Salida:", ROOT)


Salida: C:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project


## 1. Solicitud de prueba y forma de los datos

Sin `sort=price`: esta ordenación devolvía los productos más baratos aunque fueran irrelevantes. Solo se incluyen parámetros que ya conocemos; si el buscador ignora la consulta, el informe lo mostrará.

In [2]:
session = requests.Session()

def get_page(term, offset=0):
    params = dict(assortment="ES", locale="es_ES", version="v2.0.0", q=term,
                  offset=offset, fetchsize=PAGE_SIZE)
    for attempt in range(MAX_RETRIES):
        try:
            response = session.get(API_URL, params=params, timeout=TIMEOUT)
            if response.status_code in (401, 403):
                raise RuntimeError(f"HTTP {response.status_code}: acceso rechazado; no se intenta eludir restricciones.")
            if response.status_code == 429 or response.status_code >= 500:
                if attempt < MAX_RETRIES-1:
                    time.sleep(2**attempt); continue
            response.raise_for_status()
            data = response.json()
            if not isinstance(data, dict) or not isinstance(data.get("items"), list):
                raise ValueError(f"JSON inesperado: {str(data)[:350]}")
            time.sleep(DELAY)
            return data
        except (requests.RequestException, ValueError) as exc:
            if attempt == MAX_RETRIES-1: raise
            time.sleep(2**attempt)

try:
    sample = get_page("leche")
    print("Prueba OK; numFound:", sample.get("numFound"), "items:", len(sample["items"]))
    print("Clases:", pd.Series([x.get("resultClass") for x in sample["items"]]).value_counts().to_dict())
    print("Nombres:", [((x.get("gridbox") or {}).get("data") or {}).get("fullTitle") or x.get("label") for x in sample["items"][:8]])
except Exception as exc:
    sample = None
    print("PRUEBA FALLIDA:", repr(exc))


Prueba OK; numFound: 82 items: 10
Clases: {'product': 10}
Nombres: ['SILVERCREST® Espumador de leche »SMSP 500 A1«', 'SILVERCREST® Espumador de leche', 'GRAEF Espumador de leche', 'SILVERCREST® Cazo para leche de acero inoxidable', 'FIN CARRÉ Chocolate', 'BODUM Cafetera »CAFFETTIERA«', 'DELONGHI Cafetera superautomática Autentica »ETAM29.510.B«', 'LEONARDO »Tavolino« Vasos térmicos, hechos a mano.']


## 2. Extracción y limpieza de precios

Busca precios en los campos documentados en los outputs y conserva una **vista diagnóstica** para otros formatos; no convierte automáticamente cantidades de texto ambiguo. Si Lidl no proporciona €/kg o €/l y tampoco se puede leer un envase inequívoco del nombre, el valor se deja vacío.

In [3]:
def norm(s):
    s = unicodedata.normalize('NFKD', str(s or '').lower())
    s = ''.join(ch for ch in s if not unicodedata.combining(ch))
    return re.sub(r'[^a-z0-9]+',' ',s).strip()

def number(v):
    if v is None or isinstance(v,(dict,list,bool)): return None
    s = str(v).strip().replace('€','').replace(' ','').replace('\u00a0','')
    if ',' in s and '.' in s:
        if s.rfind(',') > s.rfind('.'): s=s.replace('.','').replace(',','.')
        else: s=s.replace(',','')
    else: s=s.replace(',','.')
    try: return float(s)
    except (ValueError,TypeError): return None

def dict_value(obj,key):
    return obj.get(key) if isinstance(obj,dict) else None

UNIT_MAP = {'kg':('KILO',1),'kilo':('KILO',1),'g':('KILO',.001),'gr':('KILO',.001),
            'l':('LITRO',1),'litro':('LITRO',1),'ml':('LITRO',.001),'cl':('LITRO',.01),
            'ud':('UNIDAD',1),'uds':('UNIDAD',1),'unidad':('UNIDAD',1),'unidades':('UNIDAD',1),
            'docena':('DOCENA',1),'lavados':('LAVADO',1),'rollos':('ROLLO',1)}

def reference_unit(value, unit_text):
    v=number(value)
    if v is None or not unit_text: return None,None
    m=re.fullmatch(r'\s*(\d+(?:[.,]\d+)?)?\s*([A-Za-záéíóú.]+)\s*',str(unit_text).lower())
    if not m: return None,None
    q=number(m.group(1)) if m.group(1) else 1
    mapping=UNIT_MAP.get(m.group(2).rstrip('.'))
    if not mapping or not q or q<=0: return None,None
    unit, factor=mapping
    return round(v/(q*factor),4),unit

# Variaciones ortográficas controladas. No hay sinónimos comerciales ni coincidencias por descripción.
ALIASES = {'huevos':['huevo','huevos'], 'manzana':['manzana','manzanas'],
           'platano':['platano','platanos'], 'tomate':['tomate','tomates'],
           'garbanzos':['garbanzo','garbanzos'], 'atun':['atun'],
           'yogur':['yogur','yogures','yogurt'], 'zumo':['zumo','zumos'],
           'papel':['papel'], 'higienico':['higienico']}

def title_match(term, title):
    words=set(norm(title).split())
    for token in norm(term).split():
        if not words.intersection(ALIASES.get(token,[token])): return False
    return bool(words)

def quantity_from_title(title):
    """Solo tamaño singular explícito en el título, sin multiplicadores ni varios tamaños."""
    t=norm(title)
    if re.search(r'\b\d+\s*[xX]\s*\d+',str(title),re.I): return None,None
    matches=re.findall(r'\b(\d+(?:[.,]\d+)?)\s*(kg|g|gr|l|ml|cl)\b',str(title),re.I)
    if len(matches)!=1: return None,None
    q=number(matches[0][0]); mapped=UNIT_MAP.get(matches[0][1].lower())
    if not q or not mapped: return None,None
    unit,factor=mapped
    return q*factor,unit

def flatten(item,term):
    if item.get('resultClass') != 'product': return None
    d=dict_value(item.get('gridbox'),'data') or {}
    if not isinstance(d,dict): return None
    facts=d.get('keyfacts') or {}; facts=facts if isinstance(facts,dict) else {}
    name=d.get('fullTitle') or d.get('title') or item.get('label')
    if not name: return None
    price_obj=d.get('price') or {}; price_obj=price_obj if isinstance(price_obj,dict) else {}
    bp=price_obj.get('basePrice') or {}; bp=bp if isinstance(bp,dict) else {}
    brand=d.get('brand')
    if isinstance(brand,dict): brand=brand.get('name')
    cat=facts.get('wonCategoryPrimary') or d.get('category')
    if isinstance(cat,str): cat=cat.rsplit('/',1)[-1]
    price=number(price_obj.get('price'))
    # No damos por hecho que un campo ausente sea cero o que haya oferta.
    unit_price,unit=reference_unit(bp.get('price'), bp.get('unit'))
    unit_source='api' if unit_price is not None else None
    quantity,quantity_unit=quantity_from_title(name)
    if unit_price is None and price is not None and price>0 and quantity and quantity_unit:
        unit_price=round(price/quantity,4);unit=quantity_unit;unit_source='title_inferred'
    url=d.get('canonicalUrl') or item.get('url')
    product_id=item.get('code') or d.get('productId')
    return dict(searched_item=term,supermarket='Lidl',product_id=str(product_id) if product_id else None,
                ean=d.get('ean') or facts.get('ean'),brand=brand,name=html.unescape(str(name)),
                category=cat,price=price,regular_price=price,offer_price=None,on_offer=None,
                price_per_unit=unit_price,unit=unit,unit_price_unit_type=bp.get('unit'),
                available=None,url=urljoin(BASE_URL,url) if url else None,image_url=d.get('image'),
                scraped_at=datetime.now(timezone.utc).isoformat(timespec='seconds'),
                title_match=title_match(term,name),unit_price_source=unit_source,
                raw_base_price=bp.get('price'),raw_base_unit=bp.get('unit'),
                raw_price_obj=json.dumps(price_obj,ensure_ascii=False,default=str),
                quantity_in_title=quantity,quantity_unit=quantity_unit)

if sample:
    preview=[flatten(x,'leche') for x in sample['items']]
    preview=[x for x in preview if x]
    display(pd.DataFrame(preview)[['name','price','title_match','price_per_unit','unit','unit_price_source']])


,name,price,title_match,price_per_unit,unit,unit_price_source
0,SILVERCREST® Espumador de leche »SMSP 500 A1«,17.99,True,None,None,None
1,SILVERCREST® Espumador de leche,21.99,True,None,None,None
2,GRAEF Espumador de leche,37.99,True,None,None,None
3,SILVERCREST® Cazo para leche de acero inoxidable,12.49,True,None,None,None
4,FIN CARRÉ Chocolate,0.85,False,None,None,None
5,BODUM Cafetera »CAFFETTIERA«,16.99,False,None,None,None
6,DELONGHI Cafetera superautomática Autentica »E...,399.00,False,None,None,None
7,"LEONARDO »Tavolino« Vasos térmicos, hechos a m...",8.99,False,None,None,None
8,CIEN Gel de baño,1.15,False,None,None,None
9,SILVERCREST® Cafetera de plástico »SKMK 1000 A1«,19.99,False,None,None,None


## 3. Paginación fiable y límites de extracción

`numFound` puede contar resultados que no sean productos. Continuamos aunque una página venga corta; paramos ante páginas vacías, repeticiones o límite. El informe marca `possibly_incomplete` cuando no está demostrado que se haya completado la búsqueda.

In [4]:
def fetch_term(term):
    collected=[]; page_log=[]; signatures=set(); declared=None; reason='page_limit'
    for page in range(MAX_PAGES):
        offset=page*PAGE_SIZE
        response=get_page(term,offset)
        if declared is None: declared=response.get('numFound')
        items=response['items']
        signature=tuple((str(it.get('resultClass')),str(it.get('code')),str(it.get('label'))) for it in items)
        page_log.append({'term':term,'page':page+1,'offset':offset,'items':len(items),
                         'product_items':sum(it.get('resultClass')=='product' for it in items),
                         'declared_total':declared})
        if not items: reason='empty_page';break
        if signature in signatures: reason='repeated_page';break
        signatures.add(signature)
        collected.extend(p for it in items if (p:=flatten(it,term)) is not None)
        # offset avanza por slots del buscador, no por cantidad de fichas normalizadas.
        if isinstance(declared,int) and offset+PAGE_SIZE>=declared:
            reason='declared_total_reached';break
    return collected,page_log,reason,declared

rows=[]; page_logs=[]; summaries=[]
if sample is None:
    print('Extracción suspendida: no funciona la prueba de conexión.')
else:
    for i,term in enumerate(TERMS,1):
        print(f'[{i}/{len(TERMS)}] {term}')
        try:
            products,logs,reason,declared=fetch_term(term)
            rows.extend(products);page_logs.extend(logs)
            unique_ids=len({x['product_id'] for x in products if x['product_id']})
            valid=sum(p['title_match'] and p['price'] is not None and p['price']>0 for p in products)
            summaries.append(dict(term=term,declared_total=declared,product_records=len(products),
                    unique_ids=unique_ids,matched_with_price=valid,pages=len(logs),stop_reason=reason,
                    possibly_incomplete=reason in ('page_limit','repeated_page'),status='ok'))
            print('  records:',len(products),'matched with price:',valid,'stop:',reason)
        except Exception as e:
            summaries.append(dict(term=term,status='error',message=str(e),possibly_incomplete=True))
            print('  ERROR:',str(e))
    display(pd.DataFrame(summaries))


[1/1] leche
  records: 80 matched with price: 4 stop: declared_total_reached


,term,declared_total,product_records,unique_ids,matched_with_price,pages,stop_reason,possibly_incomplete,status
0,leche,82,80,79,4,9,declared_total_reached,False,ok


## 4. CSV finales y controles de calidad

- **raw**: fichas recibidas, incluidos artículos irrelevantes.
- **rejected**: fichas descartadas con motivo.
- **processed**: coincidencia estricta en el título y precio positivo (una fila por combinación búsqueda-producto). **No es todavía una cesta validada.**
- **review_unit_prices**: precios por unidad deducidos de un tamaño del título; deben revisarse antes de usarse en optimización.
- **summary** y **pages**: trazabilidad del proceso.


In [5]:
COMMON_COLUMNS=['searched_item','supermarket','product_id','ean','brand','name','category',
                'price','regular_price','offer_price','on_offer','price_per_unit','unit',
                'unit_price_unit_type','available','url','image_url','scraped_at']
if not rows:
    print('Sin resultados; no se generan CSV de productos.')
else:
    raw=pd.DataFrame(rows).drop_duplicates(subset=['searched_item','product_id','name']).reset_index(drop=True)
    raw['rejection_reason']=''
    raw.loc[~raw['title_match'],'rejection_reason']='name_not_matching_query'
    raw.loc[raw['title_match'] & (raw['price'].isna() | (raw['price']<=0)),'rejection_reason']='missing_or_invalid_price'
    accepted=raw[raw['rejection_reason']==''].copy()
    rejected=raw[raw['rejection_reason']!=''].copy()
    processed=accepted[COMMON_COLUMNS].copy()
    paths={
      'raw':RAW_DIR/'lidl_products_raw.csv',
      'rejected':RAW_DIR/'lidl_products_rejected.csv',
      'processed':PROCESSED_DIR/'lidl_products.csv',
      'unit_review':PROCESSED_DIR/'lidl_unit_prices_to_review.csv',
      'summary':RAW_DIR/'lidl_search_summary.csv',
      'pages':RAW_DIR/'lidl_pagination_log.csv'
    }
    for kind, frame in [('raw',raw),('rejected',rejected),('processed',processed),
                       ('unit_review',accepted[accepted['unit_price_source']=='title_inferred']),
                       ('summary',pd.DataFrame(summaries)),('pages',pd.DataFrame(page_logs))]:
        frame.to_csv(paths[kind],index=False,encoding='utf-8-sig')
        print(f'{kind:12s} {len(frame):5d} -> {paths[kind]}')
    print('Aceptados:',len(accepted),'| precio unitario API:',(accepted['unit_price_source']=='api').sum(),
          '| inferido de título:',(accepted['unit_price_source']=='title_inferred').sum(),
          '| sin precio unitario:',accepted['price_per_unit'].isna().sum())
    display(accepted[['searched_item','name','price','price_per_unit','unit','unit_price_source']].head(25))
    if len(accepted)==0:
        print('ADVERTENCIA: Lidl devuelve resultados pero ninguno pasa los criterios mínimos.')


raw             79 -> C:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project\data\raw\lidl_products_raw.csv
rejected        75 -> C:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project\data\raw\lidl_products_rejected.csv
processed        4 -> C:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project\data\processed\lidl_products.csv
unit_review      0 -> C:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project\data\processed\lidl_unit_prices_to_review.csv
summary          1 -> C:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project\data\raw\lidl_search_summary.csv
pages            9 -> C:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project\data\raw\lidl_pagination_log.csv
Aceptados: 4 | precio unitario API: 0 | inferido de título: 0 | sin precio unitario: 4


,searched_item,name,price,price_per_unit,unit,unit_price_source
0,leche,SILVERCREST® Espumador de leche »SMSP 500 A1«,17.99,None,None,None
1,leche,SILVERCREST® Espumador de leche,21.99,None,None,None
2,leche,GRAEF Espumador de leche,37.99,None,None,None
3,leche,SILVERCREST® Cazo para leche de acero inoxidable,12.49,None,None,None


## 5. Criterios antes de integrar Lidl

1. Revisar `lidl_products.csv`: confirmar que los nombres corresponden a las categorías buscadas (coincidencia de título **no equivale** a producto comparable).
2. Revisar `lidl_unit_prices_to_review.csv`: los precios calculados desde el título son hipótesis y pueden ser erróneos en multipacks.
3. `lidl_search_summary.csv` indica resultados incompletos y búsquedas sin coincidencias.
4. Confirmar precios de tienda física y disponibilidad para código postal. Esta API de catálogo no los garantiza.
5. Si `numFound` es alto y el CSV contiene pocas coincidencias, no implica por sí solo un error de extracción: podría ser la relevancia real del buscador de Lidl.

**No se asignan EAN, cantidades, ofertas, ni ubicación cuando no existen en los datos.**
